# YOLO26 para principiantes

## Práctica guiada de detección de objetos — Posgrado UDEM

**Duración aproximada:** 45 minutos  
**Plataforma:** Google Colab  
**Conocimientos previos:** ninguno de Deep Learning

### ¿Qué aprenderemos?

Una fotografía puede contener personas, vehículos y otros elementos. Una persona los reconoce inmediatamente, pero una computadora necesita un modelo que le ayude.

En esta práctica enseñaremos a la computadora a:

1. encontrar objetos dentro de una imagen;
2. decir qué tipo de objeto encontró;
3. dibujar una caja alrededor del objeto.

Esta tarea se llama **detección de objetos**. Utilizaremos **YOLO26n**, un modelo pequeño y rápido. La letra `n` significa *nano*.

> **Advertencia importante:** COCO8 tiene únicamente 8 imágenes. Sirve para aprender el procedimiento, pero no demuestra que el modelo sea confiable para una aplicación real.


## Antes de programar

Pensemos en YOLO como un estudiante:

- Una **imagen** es el ejercicio que recibe.
- Una **clase** es el nombre de un objeto, como “persona” o “autobús”.
- Una **caja delimitadora** es el rectángulo que indica dónde está el objeto.
- La **confianza** expresa qué tan segura es una predicción individual.
- Una **época** es una revisión completa de los ejemplos de entrenamiento.
- La **validación** es un examen para medir lo aprendido sin modificar el modelo.

Usaremos un modelo **preentrenado**: ya observó muchas imágenes y solamente necesita un ajuste pequeño. Esta estrategia se llama **transferencia de aprendizaje**.

### Ruta de la práctica

`Preparar Colab → cargar YOLO → entrenar → validar → interpretar métricas → probar una imagen`

### Active la GPU

En Google Colab seleccione **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**. Después ejecute las celdas en orden, de arriba hacia abajo. Cada celda utiliza resultados producidos por las anteriores.

## Paso 1. Instalar YOLO

El siguiente bloque contiene una sola instrucción:

- `%pip` solicita a Colab que utilice el instalador de paquetes de Python.
- `install` indica que queremos agregar una biblioteca.
- `-q` reduce los mensajes de instalación para evitar una salida demasiado extensa.
- `ultralytics` es el paquete que contiene YOLO.
- `==8.4.116` fija una versión concreta para que todos trabajen con la misma API.

Esta instalación ocurre dentro del entorno temporal de Colab. Si el entorno se reinicia, será necesario ejecutar nuevamente esta celda.

**Resultado esperado:** la celda termina sin un mensaje rojo de error.


In [ ]:
%pip install -q ultralytics==8.4.116


**¿Qué significa el resultado anterior?** Si la instalación terminó, Colab ya puede reconocer el paquete `ultralytics`. Los mensajes amarillos suelen ser advertencias; un mensaje rojo indica que la instrucción no pudo completarse.

## Paso 2. Importar las herramientas

Instalar e importar son acciones distintas: instalar coloca una biblioteca en el entorno; importar permite utilizarla dentro del código actual.

El bloque siguiente carga seis herramientas:

- `Path`: construye rutas de carpetas y archivos sin concatenar textos manualmente.
- `matplotlib.pyplot`: crea y muestra figuras.
- `pandas`: organiza métricas en tablas.
- `display`: presenta tablas e imágenes dentro del notebook.
- `Image`: abre archivos de imagen generados por YOLO.
- `YOLO`: es la clase principal para cargar, entrenar, validar y usar el modelo.

Las instrucciones que comienzan con `from` importan una parte específica de una biblioteca. Las que comienzan con `import` cargan un módulo completo con un nombre corto, como `plt` o `pd`.

El último `print` es una comprobación sencilla: solamente se ejecuta si todas las importaciones terminaron correctamente.

**Resultado esperado:** aparecerá el mensaje “Herramientas listas”.


In [ ]:
# Herramientas que utilizaremos durante toda la práctica.
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image
from ultralytics import YOLO

print("Herramientas listas.")


**¿Qué significa el resultado anterior?** Python ya reconoce los nombres `Path`, `plt`, `pd`, `display`, `Image` y `YOLO`. Los utilizaremos sin volver a importarlos.

## Paso 3. Elegir los parámetros

Una **variable** es un nombre que guarda un valor. En esta celda utilizamos variables para reunir las decisiones principales del ejercicio:

- `MODELO = "yolo26n.pt"`: texto con el nombre de los pesos preentrenados. La extensión `.pt` corresponde a un archivo de PyTorch.
- `DATOS = "coco8.yaml"`: texto con la configuración del dataset. El archivo YAML indica dónde están las imágenes y cuáles son las clases.
- `EPOCAS = 10`: número entero que indica cuántas revisiones completas realizará el modelo.
- `TAMANO = 640`: tamaño utilizado para preparar las imágenes antes de entrar a la red.
- `PROYECTO = "/content/yolo_udem"`: carpeta temporal de Colab donde se guardarán pesos, tablas y gráficas.

Estas variables no son los millones de parámetros internos de la red neuronal. Son decisiones externas que controlan el experimento.

`print` muestra los valores para que podamos comprobar la configuración antes de gastar tiempo en el entrenamiento.

**Resultado esperado:** `yolo26n.pt`, `coco8.yaml` y `10 épocas` aparecerán en pantalla.


In [ ]:
# Configuración central del ejercicio.
MODELO = "yolo26n.pt"
DATOS = "coco8.yaml"
EPOCAS = 10
TAMANO = 640
PROYECTO = "/content/yolo_udem"

print(MODELO, DATOS, EPOCAS, "épocas")


**¿Qué significa el resultado anterior?** La receta está definida, pero el entrenamiento todavía no comienza.

## Paso 4. Cargar y entrenar el modelo

El bloque realiza tres tareas importantes.

### 1. Crear el modelo

`modelo = YOLO(MODELO)` construye un objeto YOLO usando `yolo26n.pt`. Si los pesos no existen en Colab, Ultralytics los descarga automáticamente.

### 2. Entrenar

`modelo.train(...)` inicia el aprendizaje. Sus argumentos significan:

- `data=DATOS`: utiliza COCO8.
- `epochs=EPOCAS`: ejecuta diez épocas.
- `imgsz=TAMANO`: prepara las imágenes a tamaño 640.
- `project=PROYECTO`: utiliza nuestra carpeta de trabajo.
- `name="entrenamiento"`: coloca esta ejecución en una subcarpeta reconocible.
- `exist_ok=True`: permite repetir la celda usando el mismo nombre.
- `plots=True`: solicita gráficas de pérdidas y métricas.

Durante cada época el modelo predice, compara su resultado con las etiquetas, calcula el error y modifica sus parámetros internos.

### 3. Recordar la carpeta

`modelo.trainer.save_dir` contiene la carpeta exacta creada por Ultralytics. La convertimos en un objeto `Path` para localizar después `best.pt`.

**Resultado esperado:** una fila por época con pérdidas y métricas, seguida de la ruta donde se guardó el entrenamiento.


In [ ]:
# Cargamos y ajustamos el modelo preentrenado.
modelo = YOLO(MODELO)
modelo.train(
    data=DATOS, epochs=EPOCAS, imgsz=TAMANO,
    project=PROYECTO, name="entrenamiento",
    exist_ok=True, plots=True
)

carpeta_entrenamiento = Path(modelo.trainer.save_dir)
print("Entrenamiento guardado en:", carpeta_entrenamiento)


**¿Cómo leer el resultado del entrenamiento?**

- `box_loss` mide el error en la ubicación y tamaño de las cajas.
- `cls_loss` mide el error en la clase asignada.
- Las pérdidas deberían tender a disminuir, aunque no necesariamente bajan en todas las épocas.
- Las métricas de validación deberían mejorar o estabilizarse.

YOLO guarda normalmente dos conjuntos de pesos:

- `last.pt`: estado después de la última época.
- `best.pt`: estado asociado con el mejor resultado de validación observado.

## Paso 5. Cargar `best.pt`

El operador `/` de `Path` une partes de una ruta. Por eso `carpeta_entrenamiento / "weights" / "best.pt"` localiza el archivo sin escribir manualmente toda la dirección.

`ruta_best.exists()` comprueba que el archivo esté presente. Si falta, `raise FileNotFoundError` detiene la ejecución con un mensaje comprensible, en lugar de producir un error difícil de interpretar más adelante.

Finalmente, `YOLO(str(ruta_best))` crea un modelo nuevo utilizando los mejores pesos. `str` convierte la ruta al formato de texto esperado por Ultralytics.

**Resultado esperado:** aparecerá la ruta completa del archivo `best.pt`.


In [ ]:
# Localizamos y cargamos los mejores pesos.
ruta_best = carpeta_entrenamiento / "weights" / "best.pt"
if not ruta_best.exists():
    raise FileNotFoundError("No se encontró best.pt; revise el entrenamiento.")

mejor_modelo = YOLO(str(ruta_best))
print("Mejor modelo:", ruta_best)


**¿Qué significa el resultado anterior?** `mejor_modelo` representa el checkpoint elegido y puede utilizarse para validar o predecir sin repetir el entrenamiento.

## Paso 6. Validar y obtener métricas

Validar es similar a aplicar un examen: el modelo realiza predicciones, pero ya no modifica sus parámetros.

Las cinco métricas del ejercicio son:

- **Precisión:** cuando detecta un objeto, ¿con qué frecuencia tiene razón?
- **Recall:** de todos los objetos reales, ¿cuántos encontró?
- **F1:** combina precisión y recall y penaliza que una sea muy baja.
- **mAP@0.50:** resume clasificación y localización con una coincidencia de cajas moderada.
- **mAP@0.50:0.95:** promedia varios niveles de coincidencia y es más estricta.

El bloque hace lo siguiente:

1. `mejor_modelo.val(...)` ejecuta la validación y guarda gráficas en `PROYECTO/validacion`.
2. `results_dict` proporciona precisión y recall; `float` los convierte en números comunes de Python.
3. F1 se calcula como `2 × precisión × recall / (precisión + recall)`. La condición final evita dividir entre cero.
4. `pd.DataFrame` organiza los nombres y valores en dos columnas.
5. `style.format` muestra tres decimales sin alterar los valores originales.
6. `metricas.save_dir` recuerda la ubicación exacta de las gráficas.

**Resultado esperado:** una tabla con exactamente cinco filas y la ruta de las gráficas.


In [ ]:
# Evaluamos el modelo y guardamos las gráficas en una ruta conocida.
metricas = mejor_modelo.val(
    data=DATOS, plots=True, project=PROYECTO,
    name="validacion", exist_ok=True
)

precision = float(metricas.results_dict["metrics/precision(B)"])
recall = float(metricas.results_dict["metrics/recall(B)"])
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
tabla = pd.DataFrame({
    "Métrica": ["Precisión", "Recall", "F1", "mAP@0.50", "mAP@0.50:0.95"],
    "Valor": [precision, recall, f1, metricas.box.map50, metricas.box.map]
})

display(tabla.style.format({"Valor": "{:.3f}"}))
carpeta_validacion = Path(metricas.save_dir)
print("Gráficas guardadas en:", carpeta_validacion)


**¿Cómo interpretar la tabla?**

- Una precisión alta significa pocas falsas alarmas.
- Un recall alto significa pocos objetos omitidos.
- F1 ayuda a revisar si ambas medidas están equilibradas.
- Una diferencia grande entre las dos versiones de mAP puede indicar que el modelo reconoce los objetos, pero no siempre ajusta bien las cajas.

COCO8 solo tiene cuatro imágenes de validación. Una sola imagen puede cambiar considerablemente las cifras; por ello, **estas métricas no demuestran confiabilidad en producción**.

## Paso 7. Revisar dos gráficas

`graficas` es una lista de pares. Cada par contiene un título fácil de leer y el nombre real del archivo.

El ciclo `for titulo, nombre in graficas` revisa una gráfica a la vez:

1. construye su ruta mediante `carpeta_validacion / nombre`;
2. imprime el título;
3. comprueba la existencia con `ruta.exists()`;
4. abre y muestra la imagen únicamente si existe;
5. presenta un aviso, sin detener el notebook, cuando falta un archivo.

Ultralytics utiliza `BoxPR_curve.png` para la curva precisión–recall de detección. La matriz normalizada se guarda como `confusion_matrix_normalized.png`.

**Resultado esperado:** ambas gráficas o un aviso claro para cualquier archivo no generado.


In [ ]:
# Mostramos cada gráfica solo si realmente fue creada.
graficas = [
    ("Curva precisión-recall", "BoxPR_curve.png"),
    ("Matriz de confusión", "confusion_matrix_normalized.png")
]
for titulo, nombre in graficas:
    ruta = carpeta_validacion / nombre
    print(titulo)
    if ruta.exists():
        display(Image.open(ruta))
    else:
        print("No disponible:", ruta)


**¿Cómo leer las gráficas?**

- En la curva PR buscamos un buen equilibrio entre precisión y recall. Una curva más cercana a la parte superior derecha suele ser mejor.
- En la matriz de confusión, una diagonal fuerte suele representar clasificaciones correctas. Los errores fuera de la diagonal muestran confusiones entre clases o con el fondo.

## Paso 8. Probar una imagen nueva

`mejor_modelo.predict(...)` aplica el detector a una imagen que se encuentra en internet.

- El primer argumento es la dirección de la imagen.
- `conf=0.25` descarta predicciones con confianza menor a 25 %.
- El método devuelve una lista porque podría procesar varias imágenes. `[0]` selecciona el primer resultado.
- `resultado.plot()` crea una copia de la imagen con cajas, clases y confianzas.
- Ultralytics entrega colores en orden BGR y Matplotlib espera RGB. `[..., ::-1]` invierte ese orden.
- `plt.axis("off")` oculta los ejes porque representan píxeles y no son necesarios para esta explicación.

**Resultado esperado:** una fotografía con cajas alrededor del autobús, personas u otros objetos detectados.


In [ ]:
# Detectamos objetos en una imagen nueva.
resultado = mejor_modelo.predict(
    "https://ultralytics.com/images/bus.jpg", conf=0.25
)[0]

plt.figure(figsize=(9, 7))
plt.imshow(resultado.plot()[..., ::-1])
plt.axis("off")
plt.show()


**¿Qué significa el resultado?** Cada rectángulo es una detección. El texto muestra la clase y el número asociado indica la confianza de esa predicción.

### Confianza no significa confiabilidad

- La **confianza** pertenece a una predicción individual.
- La **confiabilidad** describe el comportamiento del sistema en muchas imágenes representativas.

Una caja con confianza alta todavía puede estar equivocada. Para evaluar confiabilidad necesitamos un conjunto de prueba grande, separado del entrenamiento y parecido a las condiciones reales de uso.

### Errores que debemos buscar visualmente

- **Falso positivo:** el modelo dibuja una caja donde no existe el objeto indicado.
- **Falso negativo:** existe un objeto real, pero el modelo no lo detecta.
- **Clase incorrecta:** encuentra el objeto, pero le asigna un nombre equivocado.
- **Caja imprecisa:** la clase es correcta, pero el rectángulo no cubre adecuadamente el objeto.

### Preguntas para discutir en clase

1. ¿En qué problema de una organización sería útil detectar objetos?
2. ¿Qué sería más costoso en ese problema: una falsa alarma o no encontrar un objeto real?
3. ¿Qué imágenes deberían incluirse en una evaluación antes de utilizar el sistema?


# Conclusiones del ejercicio

## 1. Resultado técnico

Completamos el ciclo esencial de un proyecto de detección:

`instalar → configurar → entrenar → seleccionar best.pt → validar → interpretar → predecir`

La existencia de `best.pt` confirma que el entrenamiento terminó y produjo un modelo reutilizable. La imagen anotada confirma que el modelo puede recibir una entrada y devolver clases, cajas y confianzas.

## 2. Aprendizajes principales

- YOLO realiza clasificación y localización en una sola tarea.
- La transferencia de aprendizaje permite comenzar con conocimiento visual previo.
- Entrenamiento y validación tienen funciones diferentes: uno ajusta el modelo y la otra mide su comportamiento.
- Las métricas resumen resultados, pero deben complementarse con revisión visual.
- El umbral de confianza controla qué detecciones se muestran, pero no convierte al sistema en confiable.

## 3. Interpretación de las métricas

Precisión responde si las detecciones suelen ser correctas; recall responde si encontramos los objetos existentes; F1 revisa el equilibrio. mAP incorpora la capacidad de clasificar y colocar cajas correctamente bajo distintos criterios de superposición.

No existe una métrica universalmente “buena”. El valor aceptable depende del problema y del costo de falsos positivos y falsos negativos.

## 4. Limitaciones

Este ejercicio utiliza COCO8, con cuatro imágenes de entrenamiento y cuatro de validación. Además, YOLO26n parte de pesos preentrenados con COCO. Por estas razones, los resultados sirven para aprender y comprobar la tubería, pero **no permiten afirmar que el modelo esté listo para producción**.

## 5. Siguiente paso recomendado

Para un proyecto real se debe definir una clase de interés, recopilar imágenes representativas, revisar las etiquetas y separar los datos en entrenamiento, validación y prueba. La evaluación final debe realizarse sobre el conjunto de prueba una sola vez y considerar tanto métricas como errores concretos.

### Idea final

Un modelo confiable no es solamente un archivo que ejecuta predicciones: es el resultado de datos adecuados, evaluación independiente, criterios de aceptación y monitoreo continuo.

## Fuentes oficiales

- [Entrenamiento con Ultralytics YOLO](https://docs.ultralytics.com/modes/train/)
- [Validación y métricas](https://docs.ultralytics.com/modes/val/)
- [Gráficas de desempeño](https://docs.ultralytics.com/guides/yolo-performance-metrics/)
- [Dataset COCO8](https://docs.ultralytics.com/datasets/detect/coco8/)
- [Modelo YOLO26](https://docs.ultralytics.com/models/yolo26/)
